# F-09 Whisper 4차 학습 — SP 태그 수정 재전처리 + final3 이어받기

**변경 사항 (finetune_3rd.ipynb 대비)**
- 전처리: `preprocess_v2.py` SP 태그 처리 수정 — `(SP:발음)정답` → 태그 제거 후 정답만 유지
- 데이터: `processed/senior_speech_v2` 삭제 후 재전처리
- 학습: `final3` LoRA 가중치 이어받아 4차 학습 (lr=5e-6, 2000 steps)
- 저장: `final4`, 체크포인트: `stage4`

**실행 순서**: 셀 01 → 02 → 03 → 04

In [ ]:
# 셀 01 — 라이브러리 설치 + Drive 마운트 + 경로 설정
!pip install -q \
    transformers \
    datasets \
    peft \
    accelerate \
    evaluate \
    jiwer \
    librosa \
    soundfile \
    tensorboard \
    "torchao>=0.16.0"

from google.colab import drive
from pathlib import Path
import os

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_ROOT      = Path('/content/drive/MyDrive/Dadam_dataSet')
DATASET_PATH    = DRIVE_ROOT / 'processed/senior_speech_v2'
CHECKPOINT_DIR  = DRIVE_ROOT / 'checkpoints/whisper-senior'
FINAL3_DIR      = CHECKPOINT_DIR / 'final3'   # 3차 학습 결과 (이어받기 시작점)
FINAL4_DIR      = CHECKPOINT_DIR / 'final4'   # 4차 학습 결과
CHECKPOINT4_DIR = CHECKPOINT_DIR / 'stage4'
CHECKPOINT4_DIR.mkdir(parents=True, exist_ok=True)

print('완료')
print(f'데이터셋 경로 : {DATASET_PATH}')
print(f'체크포인트 경로: {CHECKPOINT_DIR}')

In [ ]:
# 셀 02 — v2 데이터셋·shard 삭제 후 재전처리
# SP 태그 처리 수정(preprocess_v2.py) 반영을 위해 기존 v2를 지우고 다시 만듦
import shutil

SHARD_V2_PATH = DRIVE_ROOT / 'processed/shards_v2'

# 기존 v2 데이터셋 삭제
if DATASET_PATH.exists():
    shutil.rmtree(str(DATASET_PATH))
    print(f'삭제 완료: {DATASET_PATH}')
else:
    print('senior_speech_v2 없음 — 건너뜀')

# 기존 shard 캐시 삭제 (재전처리 시 중복 방지)
if SHARD_V2_PATH.exists():
    shutil.rmtree(str(SHARD_V2_PATH))
    print(f'삭제 완료: {SHARD_V2_PATH}')
else:
    print('shards_v2 없음 — 건너뜀')

# 수정된 preprocess_v2.py 복사 후 실행
shutil.copy('/content/drive/MyDrive/Dadam/whisper/preprocess_v2.py', '/content/preprocess_v2.py')
print('전처리 시작 — preprocess_v2.py 실행')
!python /content/preprocess_v2.py
print('전처리 완료')

In [ ]:
# 셀 03 — 4차 학습 (final3 이어받기)
# 백그라운드 실행 활성화 후 실행할 것 (런타임 > 백그라운드 실행)
import os
import re
import torch
import evaluate
from dataclasses import dataclass
from typing import Any
from datasets import load_from_disk
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import PeftModel

MODEL_ID     = 'openai/whisper-large-v3-turbo'
EVAL_SAMPLES = 500

if not FINAL3_DIR.exists():
    raise FileNotFoundError(f'3차 학습 결과 없음: {FINAL3_DIR}')

processor  = WhisperProcessor.from_pretrained(MODEL_ID, language='Korean', task='transcribe')

# float32 로드 — Trainer fp16=True가 자동 변환
base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
base_model.generation_config.language           = 'korean'
base_model.generation_config.task               = 'transcribe'
base_model.generation_config.forced_decoder_ids = None
# suppress_tokens 건드리지 않음 — Whisper 기본값 유지 (repetition 방지)

# is_trainable=True — 4차 학습을 위해 LoRA 파라미터 그래디언트 활성화
model = PeftModel.from_pretrained(base_model, str(FINAL3_DIR), is_trainable=True)
model.print_trainable_parameters()

dataset = load_from_disk(str(DATASET_PATH))
print(dataset)

# ── 텍스트 정규화 (평가용) ───────────────────────────────────────────────────
PUNCT_PATTERN = re.compile(r'[.?!,。、]')

def clean_text(text: str) -> str:
    return PUNCT_PATTERN.sub('', text).replace(' ', '').strip()

# ── DataCollator ─────────────────────────────────────────────────────────────
@dataclass
class WhisperDataCollator:
    processor: Any

    def __call__(self, features):
        audio_arrays = [f['audio']['array'] for f in features]
        texts        = [f['text'] for f in features]
        inputs = self.processor(
            audio_arrays,
            sampling_rate=16_000,
            return_tensors='pt',
            padding='max_length',
            max_length=480_000,
            truncation=True,
        )
        labels = self.processor.tokenizer(
            texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=448,
        ).input_ids
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'input_features': inputs.input_features, 'labels': labels}

# ── CER 메트릭 ───────────────────────────────────────────────────────────────
cer_metric = evaluate.load('cer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    pred_str  = [clean_text(p) for p in pred_str]
    label_str = [clean_text(l) for l in label_str]
    return {'cer': round(cer_metric.compute(predictions=pred_str, references=label_str), 4)}

# ── 학습 설정 ─────────────────────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT4_DIR),
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,         # 유효 배치: 64
    learning_rate=5e-6,                    # final3 이어받기 — lr 더 낮춰 미세 조정
    warmup_steps=100,
    max_steps=2000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=448,
    report_to='tensorboard',
    save_total_limit=3,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'].select(range(EVAL_SAMPLES)),
    data_collator=WhisperDataCollator(processor=processor),
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

# 체크포인트 이어받기
last_checkpoint = None
checkpoints = sorted(CHECKPOINT4_DIR.glob('checkpoint-*'), key=os.path.getmtime)
if checkpoints:
    last_checkpoint = str(checkpoints[-1])
    print(f'체크포인트 발견 — 이어서 학습: {last_checkpoint}')
else:
    print('체크포인트 없음 — 처음부터 4차 학습 시작')

trainer.train(resume_from_checkpoint=last_checkpoint)

model.save_pretrained(str(FINAL4_DIR))
processor.save_pretrained(str(FINAL4_DIR))
print(f'4차 학습 완료. 저장 위치: {FINAL4_DIR}')

In [ ]:
# 셀 04 — final4 평가 (구두점+공백 제거 CER)
# 셀 03 완료 후 실행
!pip install -q evaluate jiwer

import re
import torch
import evaluate
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import Any
from pathlib import Path

# 셀 01을 실행하지 않은 경우 경로 재설정
if 'FINAL4_DIR' not in dir():
    DRIVE_ROOT     = Path('/content/drive/MyDrive/Dadam_dataSet')
    DATASET_PATH   = DRIVE_ROOT / 'processed/senior_speech_v2'
    CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints/whisper-senior'
    FINAL4_DIR     = CHECKPOINT_DIR / 'final4'

PUNCT_PATTERN = re.compile(r'[.?!,。、]')

def clean_text(text: str) -> str:
    return PUNCT_PATTERN.sub('', text).replace(' ', '').strip()

cer_metric   = evaluate.load('cer')
EVAL_SAMPLES = 500

GEN_KWARGS = dict(language='korean', task='transcribe', num_beams=1)

# 평가용 모델 로드 (셀 03을 실행하지 않은 경우)
if 'model' not in dir() or not hasattr(model, 'generate'):
    from transformers import WhisperProcessor, WhisperForConditionalGeneration
    from peft import PeftModel
    from datasets import load_from_disk
    MODEL_ID   = 'openai/whisper-large-v3-turbo'
    processor  = WhisperProcessor.from_pretrained(MODEL_ID, language='Korean', task='transcribe')
    base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    base_model.generation_config.language           = 'korean'
    base_model.generation_config.task               = 'transcribe'
    base_model.generation_config.forced_decoder_ids = None
    model = PeftModel.from_pretrained(base_model, str(FINAL4_DIR), is_trainable=False)
    model = model.to('cuda').eval()
    dataset = load_from_disk(str(DATASET_PATH))

@dataclass
class EvalCollator:
    processor: Any

    def __call__(self, features):
        audio_arrays = [f['audio']['array'] for f in features]
        texts        = [f['text'] for f in features]
        inputs = self.processor(
            audio_arrays,
            sampling_rate=16_000,
            return_tensors='pt',
            padding='max_length',
            max_length=480_000,
            truncation=True,
        )
        return {'input_features': inputs.input_features, 'texts': texts}

eval_subset = dataset['validation'].select(range(EVAL_SAMPLES))
loader      = DataLoader(eval_subset, batch_size=16, collate_fn=EvalCollator(processor))

all_preds, all_refs = [], []

for batch in loader:
    input_feats = batch['input_features'].to('cuda').half()
    with torch.no_grad():
        pred_ids = model.generate(input_features=input_feats, **GEN_KWARGS)
    preds = processor.batch_decode(pred_ids, skip_special_tokens=True)
    all_preds.extend([clean_text(p) for p in preds])
    all_refs.extend( [clean_text(r) for r in batch['texts']])

cer_result = cer_metric.compute(predictions=all_preds, references=all_refs)

print(f'final4 CER ({EVAL_SAMPLES}개): {cer_result:.4f}  →  {cer_result * 100:.2f}%')
print()
print('── 예측 vs 정답 샘플 5개 ──')
for i in range(5):
    print(f'  정답: {all_refs[i]}')
    print(f'  예측: {all_preds[i]}')
    print()

In [ ]:
# 셀 05 — v2 데이터셋 SP 태그 재전처리 여부 확인
from pathlib import Path
from datasets import load_from_disk
import re

if 'DATASET_PATH' not in dir():
    DATASET_PATH = Path('/content/drive/MyDrive/Dadam_dataSet/processed/senior_speech_v2')

print('v2 존재:', DATASET_PATH.exists())

if DATASET_PATH.exists():
    texts = load_from_disk(str(DATASET_PATH))['train'].select_columns(['text'])
    sp_pattern = re.compile(r'\(SP[: ][^)]+\)')
    found = False
    for row in texts.select(range(500)):
        if sp_pattern.search(row['text']):
            print('SP 태그 발견 — 재전처리 미완:', row['text'])
            found = True
            break
    if not found:
        print('SP 태그 없음 — 재전처리 정상 완료')